# [TIL] 2026.11.06 — FastAPI CORS와 Frontend 연동, 환경 변수 및 보안 기초

## 오늘의 학습 목표

지금까지 FastAPI를 공부하면서 API를 만들고, PostgreSQL과 연결하고, Swagger에서 요청과 응답을 테스트하는 과정을 배웠다.

하지만 실제 웹 서비스를 만들려면 백엔드 API를 프론트엔드에서도 사용할 수 있어야 한다.

오늘은 **FastAPI와 Frontend를 연결하는 방법**, 그리고 그 과정에서 필요한 **CORS, 환경 변수, 기본 보안 개념**을 학습했다.

## 1. Frontend와 Backend의 연결 구조

웹 브라우저에서 사용자가 요청하면 다음과 같은 흐름으로 데이터가 전달된다.

```text
Browser
   ↓
Frontend JavaScript
   ↓
fetch()
   ↓
CORS 검사
   ↓
FastAPI
   ↓
Service / Repository
   ↓
PostgreSQL
   ↓
JSON Response
   ↓
Browser UI
```

기존에는 Swagger를 통해 API가 정상 작동하는지만 확인했다면, 이번에는 실제 브라우저에서 API를 호출하는 과정까지 이해하는 것이 목표였다.

## 2. CORS란?

CORS(Cross-Origin Resource Sharing)는 서로 다른 출처(Origin) 사이에서 브라우저가 리소스 요청을 허용할 수 있도록 하는 규칙이다.

Origin은 다음 세 가지 요소로 구분된다.

- Protocol: http 또는 https
- Host: localhost, 도메인 등
- Port: 3000, 8000 등

예를 들어 다음 두 주소는 서로 다른 Origin이다.

```text
Frontend: http://localhost:3000
Backend:  http://localhost:8000
```

포트 번호가 다르기 때문이다.

브라우저는 동일 출처 정책(Same-Origin Policy)에 따라 다른 Origin의 응답을 JavaScript가 읽는 것을 기본적으로 제한한다.

이때 서버가 적절한 CORS 응답 헤더를 제공하면 브라우저에서 해당 응답에 접근할 수 있다.

**핵심:** CORS는 FastAPI 자체의 접근 제어 기능이라기보다 브라우저가 적용하는 보안 정책과 관련이 있다.

## 3. FastAPI에서 CORS 설정하기

FastAPI에서는 `CORSMiddleware`를 이용해 허용할 Origin을 설정할 수 있다.

```python
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:3000",
        "http://127.0.0.1:3000",
    ],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)
```

주요 설정의 의미는 다음과 같다.

| 설정 | 의미 |
|---|---|
| allow_origins | 요청을 허용할 Origin |
| allow_credentials | 쿠키 등 인증 정보가 포함된 요청 허용 여부 |
| allow_methods | 허용할 HTTP 메서드 |
| allow_headers | 허용할 요청 헤더 |

개발 과정에서 모든 Origin을 허용하면 편리할 수 있지만, 실제 운영 환경에서는 필요한 주소만 명시하는 것이 안전하다.

## 4. JavaScript fetch()로 API 호출하기

Frontend에서는 `fetch()`를 이용해 FastAPI에 HTTP 요청을 보낼 수 있다.

```javascript
async function getMembers() {
  try {
    const response = await fetch(
      "http://localhost:8000/members"
    );

    if (!response.ok) {
      throw new Error("API 요청 실패");
    }

    const members = await response.json();

    console.log(members);
  } catch (error) {
    console.error(error);
  }
}

getMembers();
```

여기서 학습한 핵심 개념은 다음과 같다.

- `fetch()` : HTTP 요청 전송
- `async/await` : 비동기 작업 처리
- `response.ok` : HTTP 응답 성공 여부 확인
- `response.json()` : JSON 응답을 JavaScript 데이터로 변환
- `try/catch` : 오류 발생 시 예외 처리

Swagger에서 직접 API를 실행하는 것과 달리, 실제 Frontend에서는 JavaScript 코드가 요청과 응답을 처리한다는 차이가 있다.

## 5. API Base URL과 환경 분리

개발 환경과 운영 환경에서는 API 주소가 달라질 수 있다.

예를 들어 개발 중에는 다음 주소를 사용한다.

```text
http://localhost:8000
```

반면 실제 서비스에서는 별도의 API 도메인을 사용할 수 있다.

```text
https://api.example.com
```

주소를 코드 곳곳에 직접 작성하면 환경이 변경될 때마다 여러 파일을 수정해야 한다.

따라서 API Base URL을 별도로 관리하는 것이 좋다.

```javascript
const API_BASE_URL = "http://localhost:8000";

async function getMembers() {
  const response = await fetch(
    `${API_BASE_URL}/members`
  );

  return await response.json();
}
```

실제 프로젝트에서는 `.env` 파일이나 프레임워크별 환경 변수 기능을 활용해 개발·운영 설정을 분리할 수 있다.

## 6. 환경 변수와 Secret 관리

오늘 배운 중요한 보안 개념 중 하나는 **브라우저에 공개해도 되는 정보와 공개하면 안 되는 정보를 구분하는 것**이다.

| 구분 | 예시 | 공개 여부 |
|---|---|---|
| 공개 설정 | Frontend API Base URL | 가능 |
| 비밀 정보 | 데이터베이스 비밀번호 | 불가 |
| 비밀 정보 | OpenAI API Key | 불가 |
| 비밀 정보 | JWT 서명용 Secret | 불가 |

Frontend 코드는 브라우저에서 실행되므로 사용자에게 노출될 수 있다.

따라서 데이터베이스 접속 정보나 API Secret을 Frontend 코드에 포함하면 안 된다.

또한 `.env` 파일을 사용하더라도 Git에 비밀 정보가 올라가지 않도록 `.gitignore` 설정을 확인해야 한다.

## 7. 오늘 배운 핵심 정리

오늘 학습한 내용을 다섯 가지로 정리했다.

1. **CORS** : 다른 Origin에서 API를 호출할 때 브라우저가 적용하는 보안 정책을 이해했다.
2. **FastAPI Middleware** : 허용할 Origin, Method, Header를 설정하는 방법을 배웠다.
3. **fetch()** : JavaScript에서 Backend API를 호출하고 JSON 응답을 처리하는 구조를 이해했다.
4. **환경 분리** : API Base URL을 개발 환경과 운영 환경에 맞게 관리해야 한다는 점을 배웠다.
5. **보안 기초** : Frontend에 공개 가능한 설정과 서버에서 보호해야 하는 Secret을 구분했다.

## 8. 오늘의 회고

지금까지는 FastAPI로 API를 만들고 Swagger에서 정상 동작하는지 확인하는 데 집중했다.

하지만 오늘 내용을 공부하면서 **API가 정상적으로 동작하는 것과 실제 웹 브라우저에서 그 API를 사용할 수 있는 것은 별개의 문제**라는 점을 알게 되었다.

특히 CORS는 처음 접하면 단순한 오류처럼 보일 수 있지만, 브라우저가 서로 다른 출처의 요청을 어떻게 처리하는지 이해하면 그 이유를 파악할 수 있다.

또한 환경 변수와 Secret 관리가 단순한 코드 정리 문제가 아니라 실제 서비스의 보안과 연결된다는 사실도 중요하게 느껴졌다.

앞으로는 API를 개발할 때 Swagger 테스트에서 끝내지 않고, 실제 Frontend에서 요청을 보내고 데이터를 화면에 표시하는 과정까지 확인하는 습관을 들이고 싶다.

**오늘의 한 줄 정리**

> Backend API를 만드는 것에서 한 걸음 더 나아가, Frontend와 안전하게 연결하기 위해 필요한 CORS와 환경 설정의 기본 구조를 배웠다.

---

**태그**

#TIL #FastAPI #CORS #Frontend #Backend